In [42]:
import numpy as np
import math
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist
from sklearn.preprocessing import OneHotEncoder

# sparse_output=False: resultaat na encoding is normale NumPy array
encoder = OneHotEncoder(sparse_output=False)

# Load mnist
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

train_images = np.resize(train_images, (60_000, 14, 14))
test_images = np.resize(test_images, (10_000, 14, 14))

# Normalize images
train_images = train_images.astype(np.float32) / 255.0
test_images = test_images.astype(np.float32) / 255.0

train_images_flat = train_images.reshape(-1, 196)
test_images_flat = test_images.reshape(-1, 196)

# One hot encode labels
train_labels_oh = encoder.fit_transform(train_labels.reshape(-1, 1))
test_labels_oh = encoder.fit_transform(test_labels.reshape(-1, 1))


In [43]:
print(train_images_flat.shape)
print(test_images_flat.shape)

(60000, 196)
(10000, 196)


In [44]:
def relu(arr):
    return np.maximum(0, arr)

def softmax(arr):
    arr = np.array(arr, dtype=np.float64)
    shifted = arr - np.max(arr, axis=-1, keepdims=True)
    exp_values = np.exp(shifted)
    return exp_values / np.sum(exp_values, axis=-1, keepdims=True)

testdata = np.array([-2.0, -0.5, 0.0, 1.5, 3.0])
batch = np.array([
    [1.0, 2.0, 3.0],
    [1.0, 0.0, -1.0]
])


print(relu(testdata))
print(softmax(batch))

print("Sum of softmax outputs for each row:")
for i in range(len(batch)):
    print(f"Row {i}: {sum(softmax(batch)[i])}")


[0.  0.  0.  1.5 3. ]
[[0.09003057 0.24472847 0.66524096]
 [0.66524096 0.24472847 0.09003057]]
Sum of softmax outputs for each row:
Row 0: 0.9999999999999999
Row 1: 0.9999999999999999


In [45]:
y_true = np.array([[0, 0, 0, 0, 0, 0, 0, 1, 0, 0]])
y_pred = np.array([[0.01, 0.02, 0.01, 0.05, 0.03, 0.02, 0.04, 0.75, 0.05, 0.02]])

def cross_entropy(y_true, y_predicted):
    e = 1e-9 # kleine waarde om log(0) te voorkomen
    return -np.mean(np.sum(y_true * np.log(y_predicted + e), axis=1))


ce_1 = cross_entropy(y_true, y_pred)
print(ce_1)


0.2876820711184476


In [46]:
input_nodes_amount = 196
hidden_nodes_amount = 128
output_nodes_amount = 10

W1 = np.random.randn(input_nodes_amount, hidden_nodes_amount) * 0.01
b1 = np.zeros(hidden_nodes_amount)

W2 = np.random.randn(hidden_nodes_amount, output_nodes_amount) * 0.01
b2 = np.zeros(output_nodes_amount)

In [47]:
def forward(X, weight1, bias1, weight2, bias2):
    Z1 = X @ weight1 + bias1
    A1 = relu(Z1)
    Z2 = A1 @ weight2 + bias2
    A2 = softmax(Z2)
    cache = (X, Z1, A1, Z2, A2)
    return (A2, cache)

input_data = train_images_flat[:5]
actual_labels = train_labels_oh[:5]

pred, cache = forward(input_data, W1, b1, W2, b2)


print(actual_labels)
# print(pred.shape)
print(pred)
# print(cache)

[[0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]]
[[0.0997054  0.09977635 0.10008957 0.10017712 0.09958646 0.10018614
  0.09995236 0.10026052 0.10034384 0.09992224]
 [0.09953144 0.09999404 0.10009244 0.10018004 0.09977014 0.10011263
  0.1000582  0.10013864 0.10003393 0.10008849]
 [0.09928252 0.099825   0.09947797 0.10069784 0.10018729 0.10028518
  0.10005051 0.10011474 0.10047028 0.09960867]
 [0.09938849 0.09981249 0.1000753  0.10012998 0.09971129 0.10021558
  0.10014739 0.09979804 0.10059604 0.10012539]
 [0.09952102 0.09964053 0.09987611 0.10043912 0.09960311 0.10019936
  0.09998486 0.10006707 0.10048772 0.10018111]]


In [48]:
def compute_output_gradient(final_prediction, correct_answers):
    N = final_prediction.shape[0] # aantal plaatjes pet batch
    # print(f"Batch size: {N}")
    return (final_prediction - correct_answers) / N

compute_output_gradient(pred, actual_labels)

array([[ 0.01994108,  0.01995527,  0.02001791,  0.02003542,  0.01991729,
        -0.17996277,  0.01999047,  0.0200521 ,  0.02006877,  0.01998445],
       [-0.18009371,  0.01999881,  0.02001849,  0.02003601,  0.01995403,
         0.02002253,  0.02001164,  0.02002773,  0.02000679,  0.0200177 ],
       [ 0.0198565 ,  0.019965  ,  0.01989559,  0.02013957, -0.17996254,
         0.02005704,  0.0200101 ,  0.02002295,  0.02009406,  0.01992173],
       [ 0.0198777 , -0.1800375 ,  0.02001506,  0.020026  ,  0.01994226,
         0.02004312,  0.02002948,  0.01995961,  0.02011921,  0.02002508],
       [ 0.0199042 ,  0.01992811,  0.01997522,  0.02008782,  0.01992062,
         0.02003987,  0.01999697,  0.02001341,  0.02009754, -0.17996378]])

In [49]:
def compute_output_gradients(hidden_output, output_gradient):
    dW2 = hidden_output.T @ output_gradient
    db2 = np.sum(output_gradient, axis=0)
    return (dW2, db2)

dw2, db2 = compute_output_gradients(cache[2], compute_output_gradient(pred, actual_labels))

print("dW2 shape:", dw2.shape)
print("db2 shape:", db2.shape)

dW2 shape: (128, 10)
db2 shape: (10,)


In [50]:
def compute_hidden_gradient(output_gradient, hidden_to_output_weights):
    return output_gradient @ hidden_to_output_weights.T

In [51]:
def relu_derivative(x):
    return (x > 0).astype(float)

In [52]:
def compute_hidden_gradients(hidden_gradient, hidden_raw_gradient, input_data):
    dZ1 = hidden_gradient * relu_derivative(hidden_raw_gradient)

    dW1 = input_data.T @ dZ1
    db1 = np.sum(dZ1, axis=0)
    return (dW1, db1)

In [53]:
def backward(y_true, cache, W2):
    # Tussenwaarden uit de forward pass uitpakken
    X, Z1, A1, Z2, A2 = cache

    # Fout bij de output: voorspelling - juiste antwoord (gemiddeld over de batch)
    output_gradient = compute_output_gradient(A2, y_true)

    # Gradient van output-gewichten W2 en bias b2
    dW2, db2 = compute_output_gradients(A1, output_gradient)

    # Foutsignaal terugsturen naar de hidden-laag via W2
    hidden_gradient = compute_hidden_gradient(output_gradient, W2)

    # Gradient van hidden-gewichten W1 en bias b1 (relu-afgeleide blokkeert inactieve nodes)
    dW1, db1 = compute_hidden_gradients(hidden_gradient, Z1, X)

    # Alle vier de gradienten teruggeven voor de update-stap
    return (dW1, db1, dW2, db2)

In [ ]:
lr = 0.1
batch_size = 128
n = train_images_flat.shape[0]
epochs = 20

print(f"Number of training samples: {n}")

for epoch in range(epochs):

    # train in batches van batch_size
    for start in range(0, n, batch_size):
        b = slice(start, start + batch_size)
        voorspelling, cache = forward(train_images_flat[b], W1, b1, W2, b2)
        dW1, db1, dW2, db2 = backward(train_labels_oh[b], cache, W2)
        W1 -= lr*dW1; b1 -= lr*db1; W2 -= lr*dW2; b2 -= lr*db2

    # einde epoch: loss/acc op de hele set
    pred_all, _ = forward(train_images_flat, W1, b1, W2, b2)
    loss = cross_entropy(train_labels_oh, pred_all)
    acc = np.mean(np.argmax(pred_all, axis=1) == train_labels)
    print(f"Epoch {epoch + 1}, Loss: {loss:.4f}, Acc: {acc:.3f}")

Number of training samples: 60000
Epoch 1, Loss: 2.3012, Acc: 0.112
Epoch 2, Loss: 2.3011, Acc: 0.113
Epoch 3, Loss: 2.3010, Acc: 0.113
Epoch 4, Loss: 2.3008, Acc: 0.113
Epoch 5, Loss: 2.3006, Acc: 0.114
Epoch 6, Loss: 2.3005, Acc: 0.114
Epoch 7, Loss: 2.3003, Acc: 0.113
Epoch 8, Loss: 2.3001, Acc: 0.113
Epoch 9, Loss: 2.2999, Acc: 0.113


In [ ]:
test_pred, _ = forward(test_images_flat, W1, b1, W2, b2)

test_loss = cross_entropy(test_labels_oh, test_pred)
test_acc = np.mean(np.argmax(test_pred, axis=1) == test_labels)

print("Final evaluation:")
print(f"Test loss: {test_loss:.4f}, Test accuracy: {test_acc:.3f}")

Final evaluation:
Test loss: 0.0878, Test accuracy: 0.975


In [ ]:
# Export weights
# destination = input("Desitination path")
# np.save(destination + "W1.npy", W1, True)
# np.save(destination + "b1.npy", b1, True)
# np.save(destination + "W2.npy", W2, True)
# np.save(destination + "b2.npy", b2, True)